In [21]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import wandb


In [22]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [23]:
wandb.init(
    project="deepfake-efficientnet-b0",
    name="efficientnet-b0-frozen-augmented w finetune",
    config={
        "architecture": "efficientnet-b0",
        "batch_size": 32,
        "epochs": 50,
        "lr": 3e-5,
        "dropout": 0.5,
        "optimizer": "AdamW",
        "loss": "BCEWithLogitsLoss"
    }
)

config = wandb.config


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/accuracy,▁▂▂▂▁▃▃▃▃▃▄▄▄▄▅▄▅▄▅▆▅▆▆▆▅▆▆▇▆▅▆▇▆▇▆▇▇▇█▇
train/f1,▁▂▃▃▂▄▄▅▅▅▆▆▅▆▆▆▇▆▆▇▆▇▇▇▇▇▇█▇▆▇▇▇▇▇█▇▇█▇
train/loss,█▇▇██▆▆▇▅▆▅▄▅▄▄▄▄▄▄▄▄▃▃▃▄▃▃▂▃▃▂▂▂▂▂▁▂▂▁▁
train/precision,▂▂▂▂▁▃▄▃▃▃▄▄▄▄▅▄▅▄▅▆▅▆▆▆▅▆▆▇▅▄▆▇▇▇▆█▆▇█▇
train/recall,▁▃▃▃▃▄▄▆▆▆▇▆▆▆▇▇█▇▇▇▆▇▇▇▇████▇██▇█▇██▇█▇
val/accuracy,▁▁▃▂▂▃▄▃▄▄▃▅▃▅▅▆▅▅▄▆▄▅▅▅▆▅▆▇▆▆▇▅▆█▆▆▆▆█▆
val/f1,▁▂▄▄▃▅▆▆▅▇▅▇▅▆▇█▇▆▆▇▆▅▇▅▇▆▆██▇▆▆▆█▆▆▇▆▇▆
val/loss,█▇▆▇▆▆▆▅▅▅▅▅▅▄▄▃▄▄▄▃▃▃▃▃▂▂▃▁▂▃▂▃▂▁▁▃▁▁▂▂
val/precision,▁▁▂▂▂▂▃▃▄▄▃▄▃▄▄▅▄▄▃▅▄▄▅▄▅▅▅▆▅▅▇▅▆▇▅▆▆▅█▆
+1,...


In [24]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),   # replaces Resize
    ##transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    #transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05)
    ),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),  # optional, adds robustness

    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [25]:
# Validation and Test transforms
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),          # Resize all images to same size
    transforms.ToTensor(),                  # Convert image to PyTorch tensor
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],        # Standard ImageNet mean
        std=[0.229, 0.224, 0.225]          # Standard ImageNet std
    )
])

In [26]:
# !mkdir data
# !unzip real_and_fake_face.zip -d data

In [27]:
DATASET_PATH = "/content/data/real_and_fake_face"

TRAIN_DIR = os.path.join(DATASET_PATH, "training")
VAL_DIR   = os.path.join(DATASET_PATH, "validation")
TEST_DIR  = os.path.join(DATASET_PATH, "testing")

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=val_test_transforms)
test_dataset  = datasets.ImageFolder(TEST_DIR, transform=val_test_transforms)

print("Class mapping:", train_dataset.class_to_idx)  # {'fake': 0, 'real': 1}

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=2)


Class mapping: {'training_fake': 0, 'training_real': 1}


In [28]:
def build_efficientnet_no_finetune():
    model = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
    )

    # Freeze entire backbone
    for param in model.features.parameters():
        param.requires_grad = False

    # Replace classifier
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, 1)
    )

    return model.to(DEVICE)

In [29]:
def build_efficientnet_finetune():
    # Load pretrained EfficientNet-B0
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    # Freeze entire backbone
    for param in model.features.parameters():
        param.requires_grad = False

    # Unfreeze last 2 MBConv blocks for fine-tuning
    for param in model.features[-2:].parameters():
        param.requires_grad = True

    # Replace classifier for binary classification
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, 1)  # 1 output for binary classification
    )

    return model.to(DEVICE)

In [30]:
criterion = nn.BCEWithLogitsLoss()

In [ ]:
def get_optimizer_no_finetune(model):
    return optim.AdamW(
        model.classifier.parameters(),
       
        lr=3e-5,
        weight_decay=1e-4
    )

In [32]:
def get_optimizer_finetune(model):
    return optim.AdamW(
        [
            {"params": model.features[-2:].parameters(), "lr": 1e-5},
            {"params": model.classifier.parameters(), "lr": 1e-4},
        ],
        weight_decay=1e-4
    )

In [33]:
from sklearn.metrics import f1_score, precision_score, recall_score
import torch
import numpy as np

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    y_true_all = []
    y_pred_all = []

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Convert logits to probabilities and predictions
        preds = (torch.sigmoid(outputs) > 0.5).int()

        # Collect for metrics
        y_pred_all.extend(preds.cpu().numpy())
        y_true_all.extend(labels.cpu().numpy().astype(int))

    # Compute accuracy
    accuracy = (np.array(y_pred_all) == np.array(y_true_all)).mean()

    # Compute F1, precision, recall
    f1 = f1_score(y_true_all, y_pred_all)
    precision = precision_score(y_true_all, y_pred_all)
    recall = recall_score(y_true_all, y_pred_all)

    avg_loss = total_loss / len(loader)

    return avg_loss, accuracy, f1, precision, recall


In [34]:
from sklearn.metrics import f1_score, precision_score, recall_score
import torch

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0

    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Convert logits to probabilities and then to predictions
            preds = (torch.sigmoid(outputs) > 0.5).int()

            # Collect for F1, precision, recall
            y_pred_all.extend(preds.cpu().numpy())
            y_true_all.extend(labels.cpu().numpy().astype(int))

    # Compute accuracy
    accuracy = (np.array(y_pred_all) == np.array(y_true_all)).mean()

    # Compute F1, precision, recall
    f1 = f1_score(y_true_all, y_pred_all)
    precision = precision_score(y_true_all, y_pred_all)
    recall = recall_score(y_true_all, y_pred_all)

    avg_loss = total_loss / len(loader)

    return avg_loss, accuracy, f1, precision, recall


In [35]:
EPOCHS = 50

# ===== Choose ONE =====
# model = build_efficientnet_no_finetune()
# optimizer = get_optimizer_no_finetune(model)

model = build_efficientnet_finetune()
optimizer = get_optimizer_finetune(model)

best_val_loss = float("inf")
patience = 7
counter = 0

for epoch in range(EPOCHS):
    # Training
    train_loss, train_acc, train_f1, train_prec, train_rec = train_one_epoch(
        model, train_loader, optimizer, criterion
    )

    # Validation
    val_loss, val_acc, val_f1, val_prec, val_rec = validate(model, val_loader, criterion)

    # Log metrics to WandB
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "train/accuracy": train_acc,
        "train/f1": train_f1,
        "train/precision": train_prec,
        "train/recall": train_rec,
        "val/loss": val_loss,
        "val/accuracy": val_acc,
        "val/f1": val_f1,
        "val/precision": val_prec,
        "val/recall": val_rec,
    })

    # Print metrics
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    print("-" * 40)

    # Early stopping and best model saving
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_efficientnet.pth")
        wandb.save("best_efficientnet.pth")
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break  # properly break the for loop


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 1/50
Train Loss: 0.7083 | Train Acc: 0.4770 | F1: 0.4971
Val   Loss: 0.6912 | Val   Acc: 0.4853 | F1: 0.5374
----------------------------------------
Epoch 2/50
Train Loss: 0.6935 | Train Acc: 0.5119 | F1: 0.5417
Val   Loss: 0.6828 | Val   Acc: 0.5441 | F1: 0.6235
----------------------------------------
Epoch 3/50
Train Loss: 0.6878 | Train Acc: 0.5389 | F1: 0.5734
Val   Loss: 0.6801 | Val   Acc: 0.5343 | F1: 0.5581
----------------------------------------
Epoch 4/50
Train Loss: 0.6784 | Train Acc: 0.5744 | F1: 0.6154
Val   Loss: 0.6759 | Val   Acc: 0.5833 | F1: 0.6559
----------------------------------------
Epoch 5/50
Train Loss: 0.6799 | Train Acc: 0.5744 | F1: 0.6225
Val   Loss: 0.6731 | Val   Acc: 0.5931 | F1: 0.6498
----------------------------------------
Epoch 6/50
Train Loss: 0.6706 | Train Acc: 0.5867 | F1: 0.6411
Val   Loss: 0.6709 | Val   Acc: 0.6029 | F1: 0.6432
----------------------------------------
Epoch 7/50
Train Loss: 0.6619 | Train Acc: 0.6130 | F1: 0.6606
V

In [36]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2

def generate_gradcam(model, input_tensor, target_layer=None):
    """
    Generate Grad-CAM heatmap for a single input image.

    Args:
        model: trained EfficientNet-B0
        input_tensor: [1, 3, H, W]
        target_layer: last convolutional layer (default: last MBConv block)
    Returns:
        heatmap: 2D numpy array (H, W) normalized 0-1
    """
    model.eval()

    # Default to last MBConv block
    if target_layer is None:
        target_layer = model.features[-1]

    # Store feature maps and gradients
    fmap_outputs = []
    grads = []

    def forward_hook(module, input, output):
        fmap_outputs.append(output)

    def backward_hook(module, grad_in, grad_out):
        grads.append(grad_out[0])

    # Register hooks
    fhook = target_layer.register_forward_hook(forward_hook)
    bhook = target_layer.register_backward_hook(backward_hook)

    # Forward pass
    output = model(input_tensor)
    model.zero_grad()

    # For binary classification: target = output for positive class
    target = output[0]
    target.backward()

    # Get feature map and gradients
    fmap = fmap_outputs[0].detach()  # shape [1, C, H, W]
    grad = grads[0].detach()         # shape [1, C, H, W]

    # Global average pooling of gradients
    weights = grad.mean(dim=(2, 3), keepdim=True)  # shape [1, C, 1, 1]

    # Weighted combination of feature maps
    cam = (weights * fmap).sum(dim=1).squeeze(0)  # shape [H, W]

    # ReLU and normalize
    cam = torch.relu(cam)
    cam = cam.cpu().numpy()
    cam = cv2.resize(cam, (input_tensor.size(3), input_tensor.size(2)))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    # Remove hooks
    fhook.remove()
    bhook.remove()

    return cam


In [50]:
print(train_dataset.class_to_idx)


{'training_fake': 0, 'training_real': 1}


In [52]:
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import cv2
import wandb
import random
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Folder to save Grad-CAM images
gradcam_dir = Path("gradcam_best_model")
gradcam_dir.mkdir(exist_ok=True)

# Load best model
best_model = build_efficientnet_finetune()
best_model.load_state_dict(torch.load("best_efficientnet.pth", map_location=DEVICE))
best_model.to(DEVICE)
best_model.eval()

# Optional: inverse normalization for visualization
inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

# Grad-CAM function
def generate_gradcam(model, input_tensor, target_layer=None, target_class=None):
    model.eval()
    if target_layer is None:
        target_layer = model.features[-1]

    fmap_outputs, grads = [], []

    def forward_hook(module, input, output):
        fmap_outputs.append(output)
    def backward_hook(module, grad_in, grad_out):
        grads.append(grad_out[0])

    fhook = target_layer.register_forward_hook(forward_hook)
    bhook = target_layer.register_backward_hook(backward_hook)

    input_tensor = input_tensor.to(next(model.parameters()).device)
    output = model(input_tensor)
    model.zero_grad()

    # If target_class is None, use predicted class
    if target_class is None:
        pred_prob = torch.sigmoid(output)
        target_class = torch.round(pred_prob)  # 0 or 1

    target = output[0] if target_class == 1 else 1 - output[0]
    target.backward()

    fmap = fmap_outputs[0].detach()
    grad = grads[0].detach()

    weights = grad.mean(dim=(2, 3), keepdim=True)
    cam = (weights * fmap).sum(dim=1).squeeze(0)
    cam = torch.relu(cam)
    cam = cam.cpu().numpy()
    cam = cv2.resize(cam, (input_tensor.size(3), input_tensor.size(2)))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    fhook.remove()
    bhook.remove()
    return cam, pred_prob.item()  # also return predicted probability

# Pick 2 fake and 2 real images
fake_indices = [i for i, (_, label) in enumerate(val_loader.dataset) if label == 0]
real_indices = [i for i, (_, label) in enumerate(val_loader.dataset) if label == 1]

sample_fake_indices = random.sample(fake_indices, min(2, len(fake_indices)))
sample_real_indices = random.sample(real_indices, min(2, len(real_indices)))
sample_indices = sample_fake_indices + sample_real_indices

for idx in sample_indices:
    image, label = val_loader.dataset[idx]
    image = image.unsqueeze(0).to(DEVICE)

    # Grad-CAM for the predicted class
    heatmap, pred_prob = generate_gradcam(best_model, image)
    pred_label = "real" if pred_prob > 0.5 else "fake"

    # Visualization
    img_vis = inv_normalize(image.squeeze(0)).cpu().permute(1, 2, 0).numpy()
    img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min())

    filename = Path(val_loader.dataset.imgs[idx][0]).name if hasattr(val_loader.dataset, "imgs") else f"image_{idx}.png"

    plt.figure(figsize=(6, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(img_vis)
    plt.title(f"Input: {filename}\nLabel: {label}, Pred: {pred_label}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(img_vis)
    plt.imshow(heatmap, cmap="jet", alpha=0.5)
    plt.title(f"Grad-CAM: {pred_label}")
    plt.axis("off")

    save_path = gradcam_dir / f"gradcam_{filename}"
    plt.savefig(save_path)
    plt.close()

    # Log to WandB
    wandb.log({f"best_model/gradcam_{filename}": wandb.Image(str(save_path))})
    print(f"Saved Grad-CAM for {filename} | Predicted: {pred_label}, Prob: {pred_prob:.4f}")

print(f"All Grad-CAM images saved to {gradcam_dir}")


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1866: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1866: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Saved Grad-CAM for mid_422_0011.jpg | Predicted: fake, Prob: 0.3703
Saved Grad-CAM for mid_405_0011.jpg | Predicted: real, Prob: 0.7293


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1866: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1866: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Saved Grad-CAM for real_00853.jpg | Predicted: real, Prob: 0.7412
Saved Grad-CAM for real_00846.jpg | Predicted: real, Prob: 0.8824
All Grad-CAM images saved to gradcam_best_model
